# 第11課 - 模型情境協定 (MCP)

The **模型情境協定 (MCP)** 是一個開放標準，使代理能在執行時動態發現並使用工具、資源與資料來源。與其將工具硬編碼到代理中，MCP 讓代理可以連接到按需公開功能的外部伺服器。

在本課中，你將學到：
- MCP 是什麼，以及它為何對代理系統重要
- MCP 的客戶端-伺服器架構如何運作
- 如何建立使用 MCP 風格工具發現的代理


## 設定

**先決條件:**
- 具有已部署模型的 Azure AI Foundry 專案
- 執行 `az login` 以進行驗證


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## 什麼是 Model Context Protocol (MCP)？

MCP 定義了一種標準方式，讓 AI 代理能夠發現並與外部工具和資料來源互動：

- **MCP Server**: 透過標準協定公開工具、資源和提示
- **MCP Client**: 連接到伺服器並發現可用功能的代理執行環境
- **Dynamic Discovery**: 代理不需要硬編碼工具 — 它們在執行時發現可用項目

這對構建可擴充的代理系統非常有用，能夠在不修改代理程式碼的情況下新增功能。


## MCP 的運作方式

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. 代理（MCP 用戶端）連線到 MCP 伺服器
2. 伺服器回應可用工具及其 schema 的清單
3. 代理在推理過程中可以呼叫任何被發現的工具
4. 結果透過相同的協定回傳


## 模擬 MCP 工具發現

由於真正的 MCP 伺服器需要一個正在執行的伺服器程序，我們將使用 `@tool` 函式來示範這個模式，模擬 MCP 連線的住宿服務會提供的功能。

在生產環境中，這些工具會從 MCP 伺服器動態發現，而不是在本地定義。


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## 建立具有 MCP 風格工具的代理人


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP 在生產環境

在生產環境中，MCP 啟用強大的模式：

- **Dynamic tool discovery**: 代理程式連接到 MCP 伺服器並在執行時發現工具
- **Decoupled architecture**: 工具提供者可以獨立於代理程式進行更新
- **Cross-organization sharing**: 團隊可以透過 MCP 伺服器公開功能，任何代理都可以使用
- **Microsoft Agent Framework support**: MAF 透過 `mcp` 整合內建 MCP 客戶端支援

若要在 MAF 中使用實際的 MCP 伺服器，您可以透過 `hosted_mcp_tool()` 或 MCP 客戶端整合來連線。

**了解更多：**
- [MCP Specification](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP Support](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## 摘要

在本課程中，您學到了：
- **MCP** 是一種用於代理與工具提供者之間動態工具發現的開放標準
- **客戶端-伺服器架構** 讓代理能在執行時發現能力
- MCP 使 **可擴充、解耦的代理系統** 成為可能，在這些系統中可以在不變更程式碼的情況下新增工具
- Microsoft Agent Framework 提供 **內建的 MCP 支援**，可用於生產環境


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
免責聲明：
本文件已使用 AI 翻譯服務 Co-op Translator（https://github.com/Azure/co-op-translator）進行翻譯。雖然我們力求準確，但請注意，自動翻譯可能包含錯誤或不精確之處。原始語言版本應視為具權威性的資料來源。對於重要資訊，建議採用專業人工翻譯。我們不對因使用本翻譯而產生的任何誤解或誤譯負責。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
